# 3D manual mock: put the laptop on charge (Canada)

The same action-grounded state machine as the triangle, but for your **laptop-charge-002** scene:
APC power strip, two loose plug heads, the MacBook adapter, the MagSafe cable, the laptop.

Press **Play** to run the manual. Three atomic MATE actions, exactly your report's procedure:
1. Attach the **North American (flat-blade)** plug head to the adapter (leave the other head).
2. Plug the adapter into an empty **outlet** on the power strip.
3. Connect the **MagSafe** connector to the laptop's charging port.

Objects are real 3D bodies (rotate / zoom with the mouse). Colors show state: grey = waiting,
orange = the part being moved now, green = connected. The red marker is the mating site for the
current step. No GPU, no model - pure geometry.

In [ ]:
!pip install -q plotly

In [ ]:
import math
import plotly.graph_objects as go

# ---------- helpers ----------
def L(a, b, t):  # lerp
    return (a[0]+(b[0]-a[0])*t, a[1]+(b[1]-a[1])*t, a[2]+(b[2]-a[2])*t)
def ADD(a, b):
    return (a[0]+b[0], a[1]+b[1], a[2]+b[2])

SX = [-1,-1, 1, 1,-1,-1, 1, 1]
SY = [-1, 1, 1,-1,-1, 1, 1,-1]
SZ = [-1,-1,-1,-1, 1, 1, 1, 1]
FI = [7,0,0,0,4,4,6,6,4,0,3,2]
FJ = [3,4,1,2,5,6,5,2,0,1,6,3]
FK = [0,7,2,3,6,7,1,1,5,5,7,6]

def box(c, s, color, opacity=1.0):
    hx, hy, hz = s[0]/2, s[1]/2, s[2]/2
    x = [c[0]+hx*v for v in SX]; y = [c[1]+hy*v for v in SY]; z = [c[2]+hz*v for v in SZ]
    return go.Mesh3d(x=x, y=y, z=z, i=FI, j=FJ, k=FK, color=color,
                     opacity=opacity, flatshading=True, hoverinfo="skip")

# ---------- fixed scene ----------
ADAPTER_REST  = (0.6, 1.5, 0.35); ADAPTER_SZ = (0.8, 0.7, 0.7)
ADAPTER_PLUG  = (-1.0, 3.4, 0.7)                 # plugged into the strip
HEAD_OFFSET   = (0.0, 0.55, 0.0)                 # head sits on the adapter back face
NA_HEAD_START = (-0.9, 0.9, 0.18); HEAD_SZ = (0.34, 0.34, 0.34)
OTHER_HEAD    = (0.25, 0.7, 0.18)                # foreign head, stays put
MAG_REST      = (2.0, 0.7, 0.12); MAG_SZ = (0.30, 0.24, 0.18)
LAP_PORT      = (2.5, 1.3, 0.2)
STRIP_C       = (-1.0, 4.05, 0.7); STRIP_SZ = (3.2, 0.5, 0.8)
OUTLETS       = [(-2.2,3.78,0.7),(-1.6,3.78,0.7),(-1.0,3.78,0.7),(-0.4,3.78,0.7),(0.2,3.78,0.7)]
TARGET_OUTLET = (-1.0, 3.78, 0.7)
LAP_BASE_C    = (3.4, 1.3, 0.07); LAP_BASE_SZ = (1.8, 1.25, 0.14)
LAP_SCR_C     = (3.4, 1.9, 0.62); LAP_SCR_SZ = (1.8, 0.12, 1.05)

ACTIONS = [
    "Step 1 / 3   |   Attach the North American flat-blade plug head to the adapter (leave the other head).   Connections: head-adapter",
    "Step 2 / 3   |   Plug the adapter into an empty outlet on the power strip.   Connections: head-adapter, adapter-strip",
    "Step 3 / 3   |   Connect the MagSafe connector to the laptop's charging port.   Connections: head-adapter, adapter-strip, cable-laptop  ->  charging",
]
C_WAIT, C_ACTIVE, C_DONE = "#bdbdbd", "#ff7f0e", "#2ca02c"

# ---------- per-frame poses ----------
def adapter_c(ph, t):
    if ph < 1: return ADAPTER_REST
    if ph == 1: return L(ADAPTER_REST, ADAPTER_PLUG, t)
    return ADAPTER_PLUG
def head_c(ph, t):
    if ph == 0: return L(NA_HEAD_START, ADD(ADAPTER_REST, HEAD_OFFSET), t)
    return ADD(adapter_c(ph, t), HEAD_OFFSET)
def mag_c(ph, t):
    if ph < 2: return MAG_REST
    return L(MAG_REST, LAP_PORT, t)

def states(ph, t):
    head = C_ACTIVE if ph == 0 else C_DONE
    adap = C_WAIT if ph == 0 else (C_ACTIVE if ph == 1 else C_DONE)
    mag  = C_WAIT if ph < 2 else (C_DONE if (ph == 2 and t >= 0.999) else C_ACTIVE)
    return head, adap, mag

def cable_xyz(ph, t):
    a = ADD(adapter_c(ph, t), (0.0, -0.35, -0.05)); m = mag_c(ph, t)
    xs, ys, zs = [], [], []
    for s in range(9):
        u = s/8.0; p = L(a, m, u)
        xs.append(p[0]); ys.append(p[1]); zs.append(max(0.04, p[2] - 0.22*math.sin(math.pi*u)))
    return xs, ys, zs

def prong_xyz(hc):
    xs, ys, zs = [], [], []
    for dx in (-0.09, 0.09):
        xs += [hc[0]+dx, hc[0]+dx, None]; ys += [hc[1]-0.17, hc[1]-0.36, None]; zs += [hc[2], hc[2], None]
    return xs, ys, zs

def build(ph, t):
    head_col, adap_col, mag_col = states(ph, t)
    ac = adapter_c(ph, t); hc = head_c(ph, t); mc = mag_c(ph, t)
    cx, cy, cz = cable_xyz(ph, t); px, py, pz = prong_xyz(hc)
    hl = [ADD(ADAPTER_REST, (0.0, 0.36, 0.0))] if ph == 0 else ([TARGET_OUTLET] if ph == 1 else [LAP_PORT])
    return [
        box((0.7, 2.4, -0.05), (8.0, 5.4, 0.08), "#b9874f", 0.35),               # desk
        box((-1.0, 4.35, 1.4), (8.0, 0.08, 3.0), "#d9d7cd", 0.30),               # back wall
        box(STRIP_C, STRIP_SZ, "#2f2f2f", 1.0),                                   # power strip
        go.Scatter3d(x=[o[0] for o in OUTLETS], y=[o[1] for o in OUTLETS], z=[o[2] for o in OUTLETS],
                     mode="markers", marker=dict(size=5, color="#111"), hoverinfo="skip", showlegend=False),
        box(LAP_BASE_C, LAP_BASE_SZ, "#c4c4cc", 1.0),                            # laptop base
        box(LAP_SCR_C, LAP_SCR_SZ, "#1c1c1c", 1.0),                             # laptop screen
        go.Scatter3d(x=[LAP_PORT[0]], y=[LAP_PORT[1]], z=[LAP_PORT[2]], mode="markers",
                     marker=dict(size=5, color="#777"), hoverinfo="skip", showlegend=False),  # port
        box(ac, ADAPTER_SZ, adap_col, 1.0),                                      # adapter
        box(hc, HEAD_SZ, head_col, 1.0),                                         # NA head
        go.Scatter3d(x=px, y=py, z=pz, mode="lines", line=dict(width=7, color="#888"),
                     connectgaps=False, hoverinfo="skip", showlegend=False),     # prongs
        box(OTHER_HEAD, HEAD_SZ, "#9aa0a6", 1.0),                               # foreign head (unused)
        box(mc, MAG_SZ, mag_col, 1.0),                                          # MagSafe connector
        go.Scatter3d(x=cx, y=cy, z=cz, mode="lines", line=dict(width=6, color="#333"),
                     hoverinfo="skip", showlegend=False),                        # cable
        go.Scatter3d(x=[h[0] for h in hl], y=[h[1] for h in hl], z=[h[2] for h in hl],
                     mode="markers", marker=dict(size=12, color="#d62728"),
                     hoverinfo="skip", showlegend=False),                        # mating-site highlight
    ]

# ---------- animation ----------
PER = 18
frames, steps = [], []
k = 0
for ph in range(3):
    for f in range(PER + 1):
        nm = "f%d" % k
        frames.append(go.Frame(data=build(ph, f/PER), name=nm, layout=go.Layout(title=ACTIONS[ph])))
        steps.append(dict(args=[[nm], dict(frame=dict(duration=0, redraw=True), mode="immediate")],
                          label="", method="animate"))
        k += 1

fig = go.Figure(
    data=build(0, 0.0),
    layout=go.Layout(
        title=ACTIONS[0],
        scene=dict(xaxis=dict(range=[-3.2, 4.8], title=""),
                   yaxis=dict(range=[0.0, 5.2], title=""),
                   zaxis=dict(range=[-0.1, 2.6], title=""),
                   aspectmode="data",
                   camera=dict(eye=dict(x=1.7, y=-1.7, z=1.1))),
        width=950, height=680,
        updatemenus=[dict(type="buttons", showactive=False, x=0, y=1,
            buttons=[dict(label="Play", method="animate",
                          args=[None, dict(frame=dict(duration=60, redraw=True), fromcurrent=True)]),
                     dict(label="Pause", method="animate",
                          args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
        sliders=[dict(active=0, steps=steps, x=0.1, len=0.85, currentvalue=dict(visible=False))]),
    frames=frames)
fig.show()

## How this is the grounded pipeline (laptop case)

- **State, not pixels:** the manual holds each object's pose and which connections exist
  (head-adapter, adapter-strip, cable-laptop). That is the structure flipbook cannot perceive.
- **Action-based steps:** every step is a MATE operator - which part moves, to which site, along which
  path - shown as motion, with the red marker as the target site.
- **Canada-specific selection:** only the flat-blade head is used; the foreign head stays grey and
  untouched, exactly the object-selection behaviour your report flagged as working.
- **Sub-assembly carry:** once the head attaches, it moves *with* the adapter in step 2 - the state
  graph knows they are one body now.
- **Verify and advance:** in a real build each step gates on the connection actually forming (pose
  check) before the next - swap the scripted motion for pose estimation to close the loop.

Scale-up: replace these boxes with the real CAD/scanned parts and the scripted target poses with the
manual's per-step poses; the action/loop/verify logic is unchanged.